# FIFA World Cup 2026 Prediction Model

**Business Question:** Which team is most likely to win the 2026 FIFA World Cup? What are the probabilities for 1st, 2nd, 3rd, and 4th place?

**Approach:**
1. Build a custom Elo rating system from 150+ years of international results
2. Train a match outcome model on Elo rating differences
3. Simulate the full 2026 tournament — group stage + knockout — 10,000 times
4. Output probabilities for Champion, Runner-up, 3rd Place, and 4th Place

**Data:** International football results 1872–present ([source](https://github.com/martj42/international_results))

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Load Data

In [ ]:
url = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'
df = pd.read_csv(url, parse_dates=['date'])

# Drop future scheduled matches (no scores yet)
df = df.dropna(subset=['home_score', 'away_score'])
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

print(f"Total matches: {len(df):,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique teams: {pd.concat([df['home_team'], df['away_team']]).nunique()}")
df.head()

## 2. Elo Rating System

Elo ratings are updated after every match. Weights account for tournament importance (K-factor), goal margin, and home advantage.

In [ ]:
def get_k_factor(tournament):
    t = tournament.lower()
    if 'fifa world cup' in t and 'qualification' not in t: return 60
    elif any(x in t for x in ['confederation','continental','copa america','euro','africa cup','gold cup','asian cup']): return 50
    elif any(x in t for x in ['qualification','qualifier']): return 40
    elif 'friendly' in t: return 20
    else: return 35

def goal_diff_multiplier(gd):
    if gd == 1: return 1.0
    elif gd == 2: return 1.5
    elif gd == 3: return 1.75
    else: return 1.75 + (gd - 3) * 0.05

def expected_result(elo_a, elo_b, home_adv=0):
    return 1 / (1 + 10 ** (-(elo_a + home_adv - elo_b) / 400))

print("Elo functions defined.")

In [ ]:
elo_ratings = {}
match_elos  = []

for _, row in df.iterrows():
    home, away = row['home_team'], row['away_team']
    neutral    = row['neutral']
    if home not in elo_ratings: elo_ratings[home] = 1500
    if away not in elo_ratings: elo_ratings[away] = 1500

    home_elo, away_elo = elo_ratings[home], elo_ratings[away]
    home_adv   = 0 if neutral else 100
    exp_home   = expected_result(home_elo, away_elo, home_adv)

    if   row['home_score'] > row['away_score']: actual = 1.0
    elif row['home_score'] == row['away_score']: actual = 0.5
    else: actual = 0.0

    gd    = abs(row['home_score'] - row['away_score'])
    delta = get_k_factor(row['tournament']) * goal_diff_multiplier(gd) * (actual - exp_home)
    elo_ratings[home] += delta
    elo_ratings[away] -= delta

    match_elos.append({'date': row['date'], 'neutral': neutral,
                       'elo_diff': home_elo - away_elo, 'result': actual})

match_df = pd.DataFrame(match_elos)
print(f"Matches processed: {len(match_df):,}")

print("\nTop 20 teams by current Elo:")
top20 = pd.Series(elo_ratings).sort_values(ascending=False).head(20)
print(top20.round(0).astype(int).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top20 = pd.Series(elo_ratings).sort_values(ascending=False).head(20)
ax.barh(top20.index[::-1], top20.values[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Elo Rating')
ax.set_title('Top 20 International Teams by Elo Rating', fontsize=14)
ax.axvline(1500, color='gray', linestyle='--', alpha=0.5, label='Average (1500)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Match Outcome Model

Logistic regression trained on Elo difference predicts win probability for any matchup. Validated on matches from 2018 onward (time-based split, no look-ahead bias).

In [ ]:
model_df = match_df[
    (match_df['date'].dt.year >= 1970) &
    (match_df['result'] != 0.5)
].copy()
model_df['home_win'] = (model_df['result'] == 1.0).astype(int)
model_df['home_adv'] = (~model_df['neutral']).astype(int)

split = pd.Timestamp('2018-01-01')
X_train = model_df[model_df['date'] < split][['elo_diff','home_adv']]
X_test  = model_df[model_df['date'] >= split][['elo_diff','home_adv']]
y_train = model_df[model_df['date'] < split]['home_win']
y_test  = model_df[model_df['date'] >= split]['home_win']

lr = LogisticRegression()
lr.fit(X_train, y_train)
y_proba = lr.predict_proba(X_test)[:, 1]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"ROC-AUC:     {roc_auc_score(y_test, y_proba):.3f}")
print(f"Brier Score: {brier_score_loss(y_test, y_proba):.3f}  (lower is better, 0.25 = random)")

## 4. 2026 World Cup — 48 Teams

The 2026 World Cup expands to 48 teams across 12 groups. Teams are seeded into 4 pots of 12 by Elo rating. Each group receives one team from each pot.

In [ ]:
wc_2026_teams = [
    # CONMEBOL (6)
    'Brazil', 'Argentina', 'Colombia', 'Uruguay', 'Ecuador', 'Venezuela',
    # UEFA (16)
    'France', 'England', 'Spain', 'Portugal', 'Netherlands', 'Germany',
    'Belgium', 'Italy', 'Croatia', 'Switzerland', 'Denmark', 'Poland',
    'Serbia', 'Austria', 'Turkey', 'Ukraine',
    # CONCACAF (6)
    'Mexico', 'United States', 'Canada', 'Panama', 'Honduras', 'Jamaica',
    # CAF (9)
    'Morocco', 'Senegal', 'Cameroon', 'Nigeria', 'Egypt',
    'Ghana', 'Tunisia', 'DR Congo', 'Algeria',
    # AFC (8)
    'Japan', 'South Korea', 'Iran', 'Australia',
    'Saudi Arabia', 'Qatar', 'Uzbekistan', 'Jordan',
    # OFC (1)
    'New Zealand',
    # Inter-confederation playoffs (2)
    'Costa Rica', 'Paraguay',
]

team_elos = {t: elo_ratings.get(t, 1500) for t in wc_2026_teams}

print(f"Total teams: {len(wc_2026_teams)}\n")
print(f"{'Team':<25} {'Elo':>6}")
print('-' * 33)
for t, e in sorted(team_elos.items(), key=lambda x: -x[1]):
    print(f"{t:<25} {int(e):>6}")

## 5. Group Stage Simulation

**Format:** 12 groups of 4 teams. Each team plays the other 3 once. Top 2 from each group (24 teams) + best 8 third-place teams (8 teams) advance to the Round of 32.

**Scoring:** Win = 3 pts, Draw = 1 pt, Loss = 0 pts. Tiebreaker: Elo rating.

**Draw probability:** Modeled as a function of Elo difference — close matches draw more often (~25% base rate), large mismatches rarely draw.

In [ ]:
def wdl_probs(elo_a, elo_b):
    """Win/Draw/Loss probabilities for team_a vs team_b on neutral ground."""
    elo_diff = elo_a - elo_b
    p_draw   = 0.25 * np.exp(-abs(elo_diff) / 500)
    p_win    = (1 / (1 + 10**(-elo_diff / 400))) * (1 - p_draw)
    p_loss   = 1 - p_win - p_draw
    return p_win, p_draw, p_loss


def sim_group_match(team_a, team_b):
    """Returns (pts_a, pts_b)."""
    p_win, p_draw, _ = wdl_probs(team_elos[team_a], team_elos[team_b])
    r = np.random.random()
    if r < p_win:               return 3, 0
    elif r < p_win + p_draw:    return 1, 1
    else:                       return 0, 3


def sim_group(group):
    """Simulate all 6 matches in a group. Returns ranked list and points dict."""
    pts = {t: 0 for t in group}
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            pa, pb = sim_group_match(group[i], group[j])
            pts[group[i]] += pa
            pts[group[j]] += pb
    ranked = sorted(group, key=lambda t: (-pts[t], -team_elos[t]))
    return ranked, pts


def assign_groups(teams, n_groups=12):
    """Pot-based seeding: top teams seeded, then one from each pot per group."""
    sorted_t = sorted(teams, key=lambda t: -team_elos[t])
    pots = [sorted_t[i*n_groups:(i+1)*n_groups] for i in range(len(teams)//n_groups)]
    groups = [[] for _ in range(n_groups)]
    for pot in pots:
        shuffled = pot[:]
        np.random.shuffle(shuffled)
        for i, t in enumerate(shuffled):
            groups[i].append(t)
    return groups


# Show one example group draw
example_groups = assign_groups(wc_2026_teams)
print("Example group draw (one simulation run):")
for i, g in enumerate(example_groups):
    elos_str = ' | '.join(f"{t} ({int(team_elos[t])})" for t in g)
    print(f"  Group {chr(65+i)}: {elos_str}")

In [ ]:
# Simulate one full group stage and show results
print("Example group stage results:\n")
for i, group in enumerate(example_groups):
    ranked, pts = sim_group(group)
    print(f"  Group {chr(65+i)}")
    for rank, team in enumerate(ranked, 1):
        qualifier = 'Q' if rank <= 2 else ' '
        print(f"    {qualifier} {rank}. {team:<22} {pts[team]} pts")
    print()

## 6. Full Tournament Simulation — 10,000 Runs

**Knockout format:** Round of 32 → Round of 16 → Quarter-finals → Semi-finals → 3rd Place match + Final.

No draws in the knockout stage (extra time / penalties modeled as a coin-flip weighted by Elo).

In [ ]:
def sim_knockout_match(team_a, team_b):
    """Returns (winner, loser) — no draws."""
    elo_diff = team_elos[team_a] - team_elos[team_b]
    p = lr.predict_proba([[elo_diff, 0]])[0][1]
    return (team_a, team_b) if np.random.random() < p else (team_b, team_a)


def sim_knockout_stage(teams_32):
    """R32 → R16 → QF → SF → 3rd place + Final. Returns (1st, 2nd, 3rd, 4th)."""
    remaining = teams_32[:]
    np.random.shuffle(remaining)

    while len(remaining) > 4:
        next_r = []
        for i in range(0, len(remaining), 2):
            w, _ = sim_knockout_match(remaining[i], remaining[i+1])
            next_r.append(w)
        remaining = next_r

    sf_w1, sf_l1 = sim_knockout_match(remaining[0], remaining[1])
    sf_w2, sf_l2 = sim_knockout_match(remaining[2], remaining[3])

    champion, runner_up = sim_knockout_match(sf_w1, sf_w2)
    third,    fourth    = sim_knockout_match(sf_l1, sf_l2)

    return champion, runner_up, third, fourth


def sim_full_tournament(teams):
    """Group stage + knockout. Returns (1st, 2nd, 3rd, 4th)."""
    groups    = assign_groups(teams)
    qualifiers = []
    third_place = []

    for group in groups:
        ranked, pts = sim_group(group)
        qualifiers.append(ranked[0])
        qualifiers.append(ranked[1])
        third_place.append((ranked[2], pts[ranked[2]]))

    best_third = sorted(third_place, key=lambda x: (-x[1], -team_elos[x[0]]))[:8]
    qualifiers += [t[0] for t in best_third]

    return sim_knockout_stage(qualifiers)


print("Simulation functions defined.")

In [ ]:
N_SIMS = 10_000
positions = {t: {1: 0, 2: 0, 3: 0, 4: 0} for t in wc_2026_teams}

for _ in range(N_SIMS):
    p1, p2, p3, p4 = sim_full_tournament(wc_2026_teams)
    positions[p1][1] += 1
    positions[p2][2] += 1
    positions[p3][3] += 1
    positions[p4][4] += 1

print(f"{N_SIMS:,} simulations complete.")

## 7. Results — Final Standings Probabilities

In [ ]:
rows = []
for t in wc_2026_teams:
    rows.append({
        'Team': t,
        'Elo': int(team_elos[t]),
        'Champion': positions[t][1] / N_SIMS,
        'Runner-up': positions[t][2] / N_SIMS,
        '3rd Place': positions[t][3] / N_SIMS,
        '4th Place': positions[t][4] / N_SIMS,
    })

results = pd.DataFrame(rows).sort_values('Champion', ascending=False).reset_index(drop=True)

display_df = results.copy()
for col in ['Champion', 'Runner-up', '3rd Place', '4th Place']:
    display_df[col] = display_df[col].map(lambda x: f'{x:.1%}')

print("2026 World Cup Final Standings Probabilities")
print(display_df.to_string(index=False))

In [ ]:
# Visualize top 12 by champion probability
top12 = results.head(12)
cols  = ['Champion', 'Runner-up', '3rd Place', '4th Place']
colors = ['gold', 'silver', '#cd7f32', '#6baed6']

fig, ax = plt.subplots(figsize=(11, 7))
bar_width = 0.18
x = np.arange(len(top12))

for i, (col, color) in enumerate(zip(cols, colors)):
    ax.bar(x + i * bar_width, top12[col], bar_width, label=col, color=color, edgecolor='white')

ax.set_xticks(x + bar_width * 1.5)
ax.set_xticklabels(top12['Team'], rotation=30, ha='right')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_title('2026 FIFA World Cup — Final Standings Probabilities (Top 12)', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Champion probability only — all teams
champ_probs = results[results['Champion'] > 0.001].set_index('Team')['Champion']

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(champ_probs.index[::-1], champ_probs.values[::-1], color='steelblue', edgecolor='white')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_title('2026 FIFA World Cup — Champion Probability', fontsize=14)
for i, v in enumerate(champ_probs.values[::-1]):
    ax.text(v + 0.001, i, f'{v:.1%}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Head-to-Head Predictor

Predict the outcome of any matchup on demand.

In [ ]:
def predict_match(team_a, team_b):
    elo_a    = elo_ratings.get(team_a, 1500)
    elo_b    = elo_ratings.get(team_b, 1500)
    elo_diff = elo_a - elo_b
    prob_a   = lr.predict_proba([[elo_diff, 0]])[0][1]
    print(f"{team_a} ({int(elo_a)}) vs {team_b} ({int(elo_b)})")
    print(f"  {team_a} wins: {prob_a:.1%}  |  {team_b} wins: {1-prob_a:.1%}")

predict_match('Spain',     'Argentina')
predict_match('Brazil',    'France')
predict_match('Argentina', 'England')
predict_match('United States', 'Mexico')

---

## Summary

**Model performance (tested on 2018–present):**
- ROC-AUC: **0.853**
- Brier Score: **0.150**

**2026 World Cup Final Standings Probabilities (10,000 simulations):**

| Team | Champion | Runner-up | 3rd Place | 4th Place |
|------|----------|-----------|-----------|----------|
| Spain | 28.0% | 10.7% | 10.8% | 2.8% |
| Argentina | 19.1% | 11.2% | 9.8% | 3.5% |
| France | 12.0% | 9.2% | 8.4% | 3.9% |
| England | 7.0% | 6.8% | 6.9% | 4.5% |
| Colombia | 4.1% | 5.3% | 5.3% | 4.6% |
| Brazil | 4.0% | 5.2% | 5.3% | 4.8% |
| Portugal | 3.9% | 5.2% | 5.1% | 4.5% |
| Ecuador | 3.3% | 4.6% | 4.7% | 4.6% |

**Key findings:**
- Spain (Elo 2153) is the clear favourite — they win the tournament in more than 1 in 4 simulations
- Spain vs Argentina account for nearly half of all simulated champions (47%)
- The group stage introduces meaningful variance: even top teams occasionally exit early
- The 2026 expanded format (48 teams, R32) slightly benefits weaker teams by adding a safety net for best 3rd-place finishers

**Limitations:**
- Group draw is randomised in simulation; actual draw will shift probabilities for specific teams
- Model does not account for injuries, squad depth, or hot/cold form streaks
- Draws are excluded from the knockout binary model (extra time / penalties treated as Elo-weighted coin flip)